# 06 · Ficha de empresa: el Bloque 2 del Company Profile

De los HTML guardados a los campos estructurados de `docs/ficha-empresa.md`.
Todo el texto es literal y cada campo guarda la URL de la que salió. No se
guardan correos ni teléfonos personales, ni cargos directivos (LOPDP).

> **Una sola fuente de verdad.** Las funciones de este notebook viven en
> `src/etl/`, se prueban con pytest y son las mismas que corre `etl` en la
> terminal y en CI. Aquí se las llama y se mira el resultado: así el notebook
> no se desactualiza cuando se corrige el código.

In [1]:
# El paquete `etl` se instala con `uv sync` en etl/.venv. Si el kernel del
# notebook no es ese entorno, se agrega src/ a la ruta como respaldo.
import sys
from pathlib import Path

ETL = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
try:
    import etl  # noqa: F401
except ModuleNotFoundError:
    sys.path.insert(0, str(ETL / "src"))

import json

import pandas as pd

from etl import config
from etl.comun import configurar_registro

configurar_registro()
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 180)

ANIO = config.ANIO_POR_DEFECTO
ANIO

2025

## 1. Correr el paso

Lee el manifiesto de capturas y los HTML de la capa crudo. No sale a internet:
se puede repetir todas las veces que haga falta mientras se afina el extractor.

In [2]:
from etl.pasos import ficha as paso_ficha

ficha = paso_ficha.ejecutar(ANIO)
display(ficha["estado"].value_counts().to_frame("empresas"))
ficha[["nombre", "estado", "tagline", "paginas_capturadas"]]

06:08:22  INFO    Ficha 2025: {'completo': 3, 'parcial': 2} -> /home/miguel/Documents/EPN/Tesis/observatorio-epn/etl/datos/plata/2025/ficha_company_profile.csv


,empresas
estado,
completo,3
parcial,2


,nombre,estado,tagline,paginas_capturadas
0,CORPORACION FAVORITA C.A.,completo,"Corporación Favorita, empresa ecuatoriana multinacional. Mejorando la calidad de vida ...",4
1,PROCESADORA NACIONAL DE ALIMENTOS C.A. PRONACA,parcial,Somos una empresa multinacional familiar que actúa con respeto para alimentar bien a m...,3
2,FERTISA AGIF C.L.,completo,"Desde 1964, FERTISA AGIF ha liderado la importación y comercialización de fertilizante...",4
3,MINUTOCORP S.A.,parcial,Minutocorp es una constructora y promotora inmobiliaria con más de 16 años aportando a...,3
4,TELCONET S.A.,completo,"Infraestructura digital y soluciones tecnológicas para empresas: fibra óptica, cable s...",4


## 2. Cada campo con su fuente

La trazabilidad no es opcional: la ficha guarda de qué URL salió cada campo
redactado, para que el asistente pueda citar.

In [3]:
# Un campo sin dato llega como NaN, no como None: `NaN or ""` devuelve NaN.
# La regla de la ficha es decirlo, no inventarlo.
def texto(valor, maximo=None):
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return "— no hay dato publicado"
    valor = str(valor)
    return valor if maximo is None or len(valor) <= maximo else valor[:maximo] + "…"


for f in ficha.itertuples():
    print(f"■ {f.nombre}   [{f.estado}]   {texto(f.url_oficial)}")
    print(f"  Tagline:     {texto(f.tagline)}")
    print(f"     ← {texto(f.tagline_fuente)}")
    print(f"  Descripción: {texto(f.descripcion, 400)}")
    print(f"     ← {texto(f.descripcion_fuente)}")
    print(f"  Productos:   {texto(f.productos_servicios)}")
    print(f"     ← {texto(f.productos_fuente)}")
    print(f"  Contacto:    {texto(f.correo_corporativo)} · {texto(f.telefono)} · {texto(f.direccion)}")
    print(f"  Redes:       {texto(f.redes)}")
    print(f"  Logo:        {texto(f.logo_url)}\n")

■ CORPORACION FAVORITA C.A.   [completo]   https://www.corporacionfavorita.com/
  Tagline:     Corporación Favorita, empresa ecuatoriana multinacional. Mejorando la calidad de vida cada día con retail, supermercados y compromiso con el…
     ← https://www.corporacionfavorita.com/
  Descripción: Trabajamos para mejorar la calidad de vida de todos, ofreciendo los mejores productos, servicios y experiencias, de forma eficiente, sostenible y responsable.

Ofrecemos productos y servicios de calidad y buscamos mejorar continuamente.

Creemos en nuestra gente, en su bienestar, felicidad y desarrollo integral.
     ← https://www.corporacionfavorita.com/quienes-somos
  Productos:   Supermercados y retail; Centros comerciales; Industria; Marcas de consumo; Construimos un gran lugar para trabajar.
     ← https://www.corporacionfavorita.com/marcas#servicios
  Contacto:    — no hay dato publicado · — no hay dato publicado · — no hay dato publicado
  Redes:       https://www.facebook.com/Corporacion

## 3. Qué campos faltan, y por qué

Un campo vacío no es un error del extractor: muchas veces la web simplemente
no publica el dato. Esta tabla dice dónde hay que mejorar y dónde no hay nada
que mejorar.

In [4]:
campos = ["tagline", "descripcion", "productos_servicios", "correo_corporativo",
          "telefono", "redes", "logo_url", "direccion"]
pd.DataFrame(
    {"con dato": [int(ficha[c].notna().sum()) for c in campos],
     "vacíos": [int(ficha[c].isna().sum()) for c in campos]},
    index=campos,
)

,con dato,vacíos
tagline,5,0
descripcion,4,1
productos_servicios,4,1
correo_corporativo,1,4
telefono,1,4
redes,2,3
logo_url,2,3
direccion,0,5


## 4. Las reglas de la LOPDP, en el código

Solo buzones institucionales; se quitan correos y teléfonos del texto y se
descartan los párrafos con cargos directivos.

In [5]:
print("correo corporativo aceptado:", paso_ficha.CORREO_CORPORATIVO.pattern)
print()
for prueba in ["info@empresa.com", "ventas@empresa.com", "juan.perez@empresa.com"]:
    print(f"  {prueba:30} {'aceptado' if paso_ficha.CORREO_CORPORATIVO.match(prueba) else 'descartado'}")
print()
texto = "El gerente general Juan Pérez explicó el plan. Escríbanos a info@empresa.com o al 0999999999."
print("limpiado:", paso_ficha.limpiar(texto))
print("párrafos útiles:", paso_ficha.parrafos_utiles(texto))

correo corporativo aceptado: ^(info|ventas|contacto|comercial|servicioalcliente|atencionalcliente|hola|hello|soporte|marketing|rrhh|talentohumano|recursoshumanos|administracion|gerencia|secretaria)@

  info@empresa.com               aceptado
  ventas@empresa.com             aceptado
  juan.perez@empresa.com         descartado

limpiado: El gerente general Juan Pérez explicó el plan. Escríbanos a o al .
párrafos útiles: []


## 5. La validación de la ficha

El tagline no puede pasar de 150 caracteres y la descripción debe tener entre
200 y 1.300. Si no se cumple, el paso se detiene.

In [6]:
print(paso_ficha.ESQUEMA)

<Schema DataFrameSchema(
    columns={
        'expediente': <Schema Column(name=expediente, type=DataType(int64))>
        'tagline': <Schema Column(name=tagline, type=DataType(string[pyarrow]))>
        'descripcion': <Schema Column(name=descripcion, type=DataType(string[pyarrow]))>
        'tagline_origen': <Schema Column(name=tagline_origen, type=DataType(string[pyarrow]))>
        'estado': <Schema Column(name=estado, type=DataType(string[pyarrow]))>
    },
    checks=[],
    parsers=[],
    coerce=False,
    dtype=None,
    index=None,
    strict=False,
    name=None,
    ordered=False,
    unique_column_names=False,
    metadata=None, 
    add_missing_columns=False,
    on_missing_columns=None
)>


## 6. Lo que sigue

Con la ficha lista, `etl consolidar` la junta con las cifras de todos los años
y `etl bd` carga todo en PostgreSQL. La API publica exactamente estas
columnas.